# Notebook 2: Embedding Creation

**Goal**: Generate vector embeddings from document chunks and analyze embedding quality.

## Topics:
- Comparing embedding models (SentenceTransformers vs OpenAI)
- Batch embedding with performance benchmarks
- Semantic similarity visualization (UMAP/PCA)
- Embedding cache hit-rate demo


In [ ]:
import sys
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from embeddings.embedding import EmbeddingEngine, SentenceTransformerEmbedder
from data_pipeline.document_loader import DocumentLoader
from data_pipeline.chunking import ChunkingEngine

print('Setup complete')

## 2.1 Prepare Documents

In [ ]:
loader = DocumentLoader()
docs = loader.load('../sample_data/', loader_type='directory')

chunker = ChunkingEngine(strategy='recursive', chunk_size=200, chunk_overlap=30)
chunks = chunker.split(docs)

print(f'Documents: {len(docs)}')
print(f'Chunks:    {len(chunks)}')
print(f'\nSample chunk:')
print(f'  Content: {chunks[0].content[:120]}...')
print(f'  Metadata: {chunks[0].metadata}')

## 2.2 Initialize Embedding Engine (Local)

In [ ]:
# Using a small local model for this demo (no API key needed)
engine = EmbeddingEngine(
    provider='sentence_transformers',
    model_name='all-MiniLM-L6-v2',   # 384-dim, lightweight, fast
    use_cache=True,
    cache_dir='../.embed_cache',
    device='cpu',
)

print(f'Embedding engine: {engine.provider}')
print(f'Dimension: {engine.dimension}')

## 2.3 Embed Chunks

In [ ]:
texts = [c.content for c in chunks]

t0 = time.perf_counter()
result = engine.encode(texts, batch_size=32)
elapsed = time.perf_counter() - t0

print(f'Embedding result:')
print(f'  Shape:       {result.embeddings.shape}')
print(f'  Model:       {result.model}')
print(f'  Dimension:   {result.dimension}')
print(f'  Latency:     {result.latency_ms:.1f} ms')
print(f'  Throughput:  {len(texts) / elapsed:.1f} texts/sec')
print(f'  Per-text:    {elapsed/len(texts)*1000:.2f} ms/text')

## 2.4 Verify Normalization

In [ ]:
vecs = result.embeddings
norms = np.linalg.norm(vecs, axis=1)

print(f'Norm statistics (should be ~1.0 for normalized vectors):')
print(f'  Mean:  {norms.mean():.6f}')
print(f'  Std:   {norms.std():.6f}')
print(f'  Min:   {norms.min():.6f}')
print(f'  Max:   {norms.max():.6f}')

assert abs(norms.mean() - 1.0) < 0.01, 'Vectors not normalized!'
print('\n✓ All vectors correctly normalized')

## 2.5 Cosine Similarity Demo

In [ ]:
# Semantic similarity between query and chunks
query = 'What is the refund policy?'
query_vec = engine.encode_query(query)

# Cosine similarity (dot product of normalized vectors)
similarities = vecs @ query_vec

# Top-5 most similar chunks
top_indices = np.argsort(similarities)[::-1][:5]

print(f'Query: "{query}"')
print('\nTop 5 most similar chunks:')
print('-' * 70)
for i, idx in enumerate(top_indices):
    print(f'[{i+1}] Score: {similarities[idx]:.4f}')
    print(f'    {texts[idx][:120]}...')
    print()

## 2.6 Similarity Heatmap

In [ ]:
# Compare a few representative sentences
sentences = [
    'Refund policy for purchases',
    'How to return a product',
    'Laptop specifications and price',
    'SmartWatch features',
    'How to track my order',
    'Payment methods accepted',
    'International shipping information',
]

sent_result = engine.encode(sentences)
sent_vecs = sent_result.embeddings

# Compute pairwise cosine similarity
sim_matrix = sent_vecs @ sent_vecs.T

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(sentences)))
ax.set_yticks(range(len(sentences)))
short = [s[:25] for s in sentences]
ax.set_xticklabels(short, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(short, fontsize=9)
plt.colorbar(im, label='Cosine Similarity')
ax.set_title('Pairwise Semantic Similarity', fontsize=13)

for i in range(len(sentences)):
    for j in range(len(sentences)):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('similarity_heatmap.png', dpi=100)
plt.show()

## 2.7 PCA Visualization of Chunk Embeddings

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
coords = pca.fit_transform(vecs)

# Color by source file
sources = [c.metadata.get('source', '?').split('\\')[-1].split('/')[-1] for c in chunks]
unique_sources = list(set(sources))
colors = plt.cm.Set1(np.linspace(0, 1, len(unique_sources)))
color_map = {s: colors[i] for i, s in enumerate(unique_sources)}

fig, ax = plt.subplots(figsize=(10, 7))
for src in unique_sources:
    mask = [s == src for s in sources]
    xs = coords[mask, 0]
    ys = coords[mask, 1]
    ax.scatter(xs, ys, label=src, alpha=0.8, s=60)

ax.set_title(f'PCA of Chunk Embeddings\n(explained variance: {pca.explained_variance_ratio_.sum():.1%})')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('pca_embeddings.png', dpi=100)
plt.show()
print('PCA saved. Notice how chunks from the same source cluster together.')

## 2.8 Cache Performance Demo

In [ ]:
# First run: cold cache
import time
test_texts = texts[:10]

t0 = time.perf_counter()
r1 = engine.encode(test_texts)
cold_ms = (time.perf_counter() - t0) * 1000

# Second run: warm cache
t0 = time.perf_counter()
r2 = engine.encode(test_texts)
warm_ms = (time.perf_counter() - t0) * 1000

print(f'Cold cache (first run):  {cold_ms:.1f} ms')
print(f'Warm cache (second run): {warm_ms:.1f} ms')
print(f'Speedup:                 {cold_ms/max(warm_ms, 0.1):.1f}x')

# Verify results are identical
max_diff = np.abs(r1.embeddings - r2.embeddings).max()
print(f'\nMax diff between runs: {max_diff:.2e} (should be 0)')

## Model Comparison Reference

| Model | Dim | Latency (CPU/batch) | MTEB Score | Cost |
|---|---|---|---|---|
| all-MiniLM-L6-v2 | 384 | ~50ms/batch | 56.3 | Free |
| BAAI/bge-small-en-v1.5 | 384 | ~70ms/batch | 62.2 | Free |
| BAAI/bge-base-en-v1.5 | 768 | ~180ms/batch | 64.2 | Free |
| BAAI/bge-large-en-v1.5 | 1024 | ~400ms/batch | 64.6 | Free |
| text-embedding-3-small | 1536 | ~100ms/API | 62.3 | $0.02/1M tokens |
| text-embedding-3-large | 3072 | ~200ms/API | 64.6 | $0.13/1M tokens |

**Rule of thumb**: For most RAG systems, `BAAI/bge-base-en-v1.5` gives 95% of the quality at 50% of the latency of the large model. Use `text-embedding-3-large` only when retrieval quality is paramount and you have API budget.